# A/B Testing: Credit Card Application (5-page vs 3-page)

**Goal:** Check if the new 3-page application increases completion rate, and make sure fraud doesn't cross the 1.5% limit.

**Data:** 15,000 users per version (A = old 5-page form, B = new 3-page form)


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats


## 1. Load the data

In [2]:
df = pd.read_csv("../data/experiment_data.csv")
df


,version,description,started_applications,completed_applications,fraudulent_applications
0,A,Existing 5-page application (Control),15000,1800,18
1,B,Simplified 3-page application (Treatment),15000,2040,25


## 2. Completion rate for each version

In [3]:
df["completion_rate"] = df["completed_applications"] / df["started_applications"]
df[["version", "completion_rate"]]


,version,completion_rate
0,A,0.120
1,B,0.136


In [4]:
# pulling out the numbers we need so the code below is easier to read
rate_a = df.loc[df["version"] == "A", "completion_rate"].values[0]
rate_b = df.loc[df["version"] == "B", "completion_rate"].values[0]

completed_a = df.loc[df["version"] == "A", "completed_applications"].values[0]
completed_b = df.loc[df["version"] == "B", "completed_applications"].values[0]

started_a = df.loc[df["version"] == "A", "started_applications"].values[0]
started_b = df.loc[df["version"] == "B", "started_applications"].values[0]

print(f"Version A completion rate: {rate_a:.2%}")
print(f"Version B completion rate: {rate_b:.2%}")


Version A completion rate: 12.00%
Version B completion rate: 13.60%


In [5]:
absolute_lift = rate_b - rate_a
relative_lift = absolute_lift / rate_a

print(f"Absolute lift: {absolute_lift:.2%}")
print(f"Relative lift: {relative_lift:.2%}")


Absolute lift: 1.60%
Relative lift: 13.33%


B has a higher completion rate than A. Now let's check if this difference is actually significant, or if it could just be random noise.

## 3. Is the difference statistically significant?

- H0 (null): There is no real difference between A and B
- H1 (alt): B has a higher completion rate than A

Using a two-proportion Z-test at 95% confidence.


In [6]:
# pooled proportion = combine both groups as if they were one group
pooled_p = (completed_a + completed_b) / (started_a + started_b)

# standard error of the difference between the two proportions
se = np.sqrt(pooled_p * (1 - pooled_p) * (1/started_a + 1/started_b))

z_score = (rate_b - rate_a) / se
p_value = 1 - stats.norm.cdf(z_score)

print(f"Pooled proportion: {pooled_p:.4f}")
print(f"Standard error: {se:.5f}")
print(f"Z-score: {z_score:.2f}")
print(f"P-value: {p_value:.6f}")


Pooled proportion: 0.1280
Standard error: 0.00386
Z-score: 4.15
P-value: 0.000017


In [7]:
z_critical = 1.645  # 95% confidence, one-tailed

if z_score > z_critical:
    print(f"Z-score ({z_score:.2f}) > critical value ({z_critical}) -> reject H0")
    print("The improvement in completion rate is statistically significant.")
else:
    print("We cannot reject H0. The difference could be due to chance.")


Z-score (4.15) > critical value (1.645) -> reject H0
The improvement in completion rate is statistically significant.


## 4. Guardrail check: did fraud go up too much?

Completion rate going up is great, but the Risk team wants to make sure fraud isn't creeping up because of the simpler form.


In [8]:
df["fraud_rate"] = df["fraudulent_applications"] / df["completed_applications"]
df[["version", "fraud_rate"]]


,version,fraud_rate
0,A,0.010000
1,B,0.012255


In [9]:
fraud_rate_a = df.loc[df["version"] == "A", "fraud_rate"].values[0]
fraud_rate_b = df.loc[df["version"] == "B", "fraud_rate"].values[0]

fraud_abs_increase = fraud_rate_b - fraud_rate_a
fraud_relative_increase = fraud_abs_increase / fraud_rate_a

print(f"Fraud rate A: {fraud_rate_a:.2%}")
print(f"Fraud rate B: {fraud_rate_b:.2%}")
print(f"Absolute increase: {fraud_abs_increase:.2%} pts")
print(f"Relative increase: {fraud_relative_increase:.2%}")


Fraud rate A: 1.00%
Fraud rate B: 1.23%
Absolute increase: 0.23% pts
Relative increase: 22.55%


In [10]:
fraud_threshold = 0.015  # bank policy: fraud rate must stay below 1.5%

if fraud_rate_b < fraud_threshold:
    print(f"Fraud rate ({fraud_rate_b:.2%}) is below the {fraud_threshold:.1%} threshold. Guardrail passed.")
else:
    print(f"Fraud rate ({fraud_rate_b:.2%}) breaches the {fraud_threshold:.1%} threshold. Guardrail failed.")


Fraud rate (1.23%) is below the 1.5% threshold. Guardrail passed.


## 5. Business impact if we roll out version B to everyone

Assumptions:
- 1,000,000 visitors/month
- Approval rate: 30%
- Profit per approved card: ₹8,000


In [11]:
monthly_visitors = 1_000_000
approval_rate = 0.30
profit_per_approved = 8_000  # in INR

current_completions = monthly_visitors * rate_a
new_completions = monthly_visitors * rate_b
extra_completions = new_completions - current_completions

extra_approved = extra_completions * approval_rate
extra_profit = extra_approved * profit_per_approved

print(f"Current completions/month: {current_completions:,.0f}")
print(f"New completions/month: {new_completions:,.0f}")
print(f"Extra completions/month: {extra_completions:,.0f}")
print(f"Extra approved customers/month: {extra_approved:,.0f}")
print(f"Extra profit/month: ₹{extra_profit/1e7:.2f} crore")


Current completions/month: 120,000
New completions/month: 136,000
Extra completions/month: 16,000
Extra approved customers/month: 4,800
Extra profit/month: ₹3.84 crore


## 6. Conclusion

- Completion rate improved from 12.0% to 13.6% (13.3% relative lift), and this is statistically significant (Z = 4.15).
- Fraud rate increased from 1.00% to 1.23%, but stays under the 1.5% policy limit.
- Recommendation: **roll out the 3-page application to all users**, and keep monitoring the fraud rate after launch.
